# 📧 업무 메일 자동 분류 시스템
## 전체 흐름
```
1. IMAP 접속
2. 메일 가져오기
3. 제목 / 발신자 / 본문 / PDF 텍스트 추출
4. 형태소 분석기 4개 비교 → 최적 분석기 선택 → 정보 추출
5. 메일별 워드클라우드 + PDF 카테고리 이동 + CSV 저장
```

---
# 1편. IMAP 접속 확인

## 라이브러리 설치

In [ ]:
# !pip install pdfplumber konlpy wordcloud scikit-learn nltk

## 계정 입력

In [ ]:
EMAIL = "dudhrdl123@gmail.com"
PASSWORD = "fahc ksan cbxm uypl"  # 앱 비밀번호 16자리

## Gmail 접속

In [ ]:
import imaplib

mail = imaplib.IMAP4_SSL("imap.gmail.com")
mail.login(EMAIL, PASSWORD)
print("✅ 접속 성공")

---
# 2편. 메일 가져오기

## 메일 기간 설정 및 목록 조회

In [ ]:
from datetime import datetime, timedelta

mail.select("INBOX")

# ── 전체 메일 ──────────────────────────────
_, msg_nums = mail.search(None, "ALL")

# ── 최근 1일 ──────────────────────────────
# since_date = (datetime.now() - timedelta(days=1)).strftime("%d-%b-%Y")
# _, msg_nums = mail.search(None, f'SINCE {since_date}')

# ── 최근 7일 ──────────────────────────────
# since_date = (datetime.now() - timedelta(days=7)).strftime("%d-%b-%Y")
# _, msg_nums = mail.search(None, f'SINCE {since_date}')

num_list = msg_nums[0].split()
print(f"📧 메일 수: {len(num_list)}건")

## 디코딩 함수 정의

In [ ]:
import email
import pandas as pd
from email.header import decode_header

# 디코딩 함수 : 제목, 발신자, 파일명 등 바이트 → 문자열
def decode_str(s):
    if s is None:
        return ""
    result = ""
    for part, enc in decode_header(s):
        if isinstance(part, bytes):
            result += part.decode(enc or "utf-8", errors="ignore")
        else:
            result += str(part)
    return result

---
# 3편. 텍스트 추출

## PDF 텍스트 추출 함수

In [ ]:
import pdfplumber
import os

def extract_pdf_text(filepath):
    text = ""
    try:
        with pdfplumber.open(filepath) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
    except Exception as e:
        text = f"[추출 실패: {e}]"
    return text.strip()

## 메일 본문 추출 함수

In [ ]:
def get_body(msg):
    for part in msg.walk():
        if part.get_content_type() == "text/plain":
            if "attachment" not in str(part.get("Content-Disposition", "")):
                try:
                    return part.get_payload(decode=True).decode("utf-8", errors="ignore")
                except:
                    pass
    return ""

## PDF 다운로드 함수

In [ ]:
DOWNLOAD_DIR = "downloads"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

def get_pdfs(msg):
    pdf_text, pdfs = "", []
    for part in msg.walk():
        if "attachment" in str(part.get("Content-Disposition", "")):
            filename = part.get_filename()
            if filename:
                filename = decode_str(filename)
                if filename.lower().endswith(".pdf"):
                    filepath = os.path.join(DOWNLOAD_DIR, filename)
                    with open(filepath, "wb") as f:
                        f.write(part.get_payload(decode=True))
                    pdf_text += extract_pdf_text(filepath)
                    pdfs.append(filepath)
    return pdf_text, pdfs

## 메일 데이터 수집

In [ ]:
results = []

for num in num_list[::-1]:
    _, data = mail.fetch(num, "(RFC822)")
    msg = email.message_from_bytes(data[0][1])
    pdf_text, pdfs = get_pdfs(msg)
    results.append({
        "subject":   decode_str(msg["Subject"]),   # 제목
        "sender":    decode_str(msg["From"]),      # 발신자
        "body":      get_body(msg).strip(),        # 본문
        "pdf_text":  pdf_text.strip(),             # PDF 텍스트
        "pdf_paths": pdfs,                         # PDF 저장 경로 목록
    })

print(f"✅ {len(results)}건 수집 완료")

## 결과 확인

In [ ]:
df_check = pd.DataFrame(results)
df_check["body_len"]  = df_check["body"].apply(len)
df_check["pdf_len"]   = df_check["pdf_text"].apply(len)
df_check["pdf_count"] = df_check["pdf_paths"].apply(len)
print(df_check[["subject", "sender", "body_len", "pdf_len", "pdf_count"]])

## 접속 종료

In [ ]:
mail.logout()
print("👋 접속 종료")

---
# 4편. 형태소 분석기 비교 및 분류

## 형태소 분석기 로드

In [ ]:
from konlpy.tag import Okt, Komoran, Kkma, Hannanum

okt      = Okt()
komoran  = Komoran()
kkma     = Kkma()
hannanum = Hannanum()

print("✅ 분석기 로드 완료")

## 불용어 및 카테고리 키워드 정의

In [ ]:
STOPWORDS = set([
    "합니다", "있습니다", "드립니다", "바랍니다", "부탁드립니다", "니다", "이다",
    "안녕하세요", "감사합니다", "관련", "내용", "진행", "경우", "이번",
    "해당", "통해", "위해", "대해", "파일", "것", "수", "때", "후", "전", "중", "등", "및",
    # 너무 범용적이라 특정 카테고리를 과하게 올릴 수 있는 단어
    "시스템", "서비스", "기능", "데이터", "결과", "분석", "확인"
])

# 카테고리별 키워드에 가중치를 부여
# - 파일명/제목에 등장하면 더 강하게 반영
# - '분석', '결과', '데이터'처럼 범용적인 단어는 제거하거나 낮게 반영
CATEGORY_WEIGHTS = {
    "업무협조": {
        "업무협조": 8, "협조": 5, "요청": 4, "검토": 4, "회신": 4,
        "확인": 2, "승인": 3, "처리": 3, "지원": 3, "일정": 2,
        "테스트": 2, "계정": 2, "연동": 3
    },
    "보고서": {
        "보고서": 9, "보고": 6, "현황": 5, "지표": 5, "통계": 5,
        "집계": 4, "매출": 4, "성과": 4, "오류율": 5, "전월": 4,
        "요약": 2, "리포트": 6
    },
    "회의록": {
        "회의록": 10, "회의": 6, "안건": 6, "논의": 6, "참석자": 6,
        "결정": 5, "결정사항": 7, "액션아이템": 7, "회의결과": 6,
        "검토회의": 8
    },
    "공지": {
        "공지": 9, "공지사항": 8, "안내문": 7, "안내": 5, "알림": 5,
        "점검": 7, "변경": 4, "제한": 4, "운영팀": 4, "배포": 4,
        "개정": 5, "처리방침": 5
    },
    "기타": {
        "기타": 5, "교육": 9, "자료": 6, "가이드": 8, "학습": 6,
        "참고": 5, "공유": 3, "라이브러리": 5, "오픈소스": 7,
        "매뉴얼": 7, "사용법": 6
    }
}

# 기존 코드 호환용: 키워드 리스트 형태도 유지
CATEGORIES = {cat: list(weights.keys()) for cat, weights in CATEGORY_WEIGHTS.items()}

TAG_MAP = {
    "[업무협조]": "업무협조",
    "[보고서]":   "보고서",
    "[회의록]":   "회의록",
    "[공지]":     "공지",
    "[자료 공유]": "기타",
    "[참고]":     "기타",
    "[안내]":     "공지",
}

def get_pdf_filename_text(pdf_paths):
    """첨부 PDF 경로 목록에서 파일명 텍스트만 추출"""
    return " ".join(os.path.basename(p) for p in pdf_paths or [])

def metadata_bonus(subject="", filename=""):
    """제목/파일명에 포함된 카테고리 키워드는 본문보다 강하게 가산"""
    scores = {cat: 0.0 for cat in CATEGORY_WEIGHTS}
    subject = subject or ""
    filename = filename or ""
    for cat, weights in CATEGORY_WEIGHTS.items():
        for kw, w in weights.items():
            if kw in subject:
                scores[cat] += w * 4
            if kw in filename:
                scores[cat] += w * 5
    return scores

def choose_best_category(scores):
    """점수가 가장 높은 카테고리와 신뢰도 반환"""
    total = sum(scores.values())
    if total == 0:
        return "기타", 0.0
    best = max(scores, key=scores.get)
    confidence = round(scores[best] / total, 2)
    return best, confidence


## 명사 추출 함수

In [ ]:
def extract_nouns(text, analyzer):
    if not text or not text.strip():
        return []
    try:
        # Kkma, Komoran → NNG(일반명사)만 추출
        if isinstance(analyzer, (Kkma, Komoran)):
            tagged = analyzer.pos(text)
            nouns = [word for word, tag in tagged if tag == "NNG" and len(word) >= 2]
        # Okt, Hannanum → nouns() 그대로 사용
        else:
            nouns = analyzer.nouns(text)
            nouns = [n for n in nouns if len(n) >= 2]

        nouns = [n for n in nouns if n not in STOPWORDS]
        return nouns
    except Exception:
        return []

### 분석기별 명사 추출 테스트

In [ ]:
sample = "업무협조 요청드립니다. 첨부파일 검토 후 회신 부탁드립니다."
print(f"Okt      : {extract_nouns(sample, okt)}")
print(f"Komoran  : {extract_nouns(sample, komoran)}")
print(f"Kkma     : {extract_nouns(sample, kkma)}")
print(f"Hannanum : {extract_nouns(sample, hannanum)}")

## 회신 / 기한 / 요약 추출 함수

In [ ]:
import re

ACTION_KEYWORDS = [
    "부탁드립니다", "확인 바랍니다", "회신 바랍니다",
    "검토 부탁", "처리 부탁", "승인 요청", "확인 부탁", "회신바랍니다"
]

# 기한 관련 키워드
DEADLINE_KEYWORDS = [
    "기한", "마감", "까지", "예정", "일정", "다음", "앞으로"
]

# 날짜 패턴
DATE_PATTERNS = [
    r"\d+/\d+\s*\(.*?\)",           # 5/13(수)
    r"\d{4}-\d{2}-\d{2}",           # 2026-05-01
    r"\d{4}년\s*\d{1,2}월\s*\d{1,2}일",  # 2026년 5월 1일
    r"\d{1,2}월\s*\d{1,2}일",        # 5월 13일
    r"\d+일\s*내",                    # 3일 내
    r"금주\s*내",
    r"이번\s*주\s*내",
    r"오늘\s*중",
    r"내일까지",
    r"긴급|즉시|빠른\s*시일",
]

def detect_action(text):
    return "필요" if any(kw in text for kw in ACTION_KEYWORDS) else "불필요"

def find_deadline(text):
    # 문장 단위로 분리
    sentences = re.split(r'[.\n]', text)

    for sentence in sentences:
        # 기한 키워드가 포함된 문장인지 확인
        if any(kw in sentence for kw in DEADLINE_KEYWORDS):
            # 해당 문장에서 날짜 추출
            for pattern in DATE_PATTERNS:
                m = re.search(pattern, sentence)
                if m:
                    return m.group(0)
    return "-"

def make_summary(body):
    sentences = [s.strip() for s in re.split(r'[.\n]', body) if len(s.strip()) > 5]
    return " / ".join(sentences[:2]) if sentences else "-"

## 방법 1: KoNLPy + FreqDist (빈도 분석) 분류 함수

In [ ]:
from nltk import FreqDist

def classify_freqdist(text, analyzer, subject="", filename=""):
    """
    명사 추출 → FreqDist 빈도 분석
    → 카테고리별 키워드 가중치 합산
    → 제목/파일명 키워드 보정 후 가장 높은 카테고리로 분류
    """

    # 제목 태그 우선 확인
    for tag, cat in TAG_MAP.items():
        if tag in subject:
            return cat, 1.0

    nouns = extract_nouns(text, analyzer)
    fd = FreqDist(nouns)

    scores = metadata_bonus(subject, filename)
    for cat, weights in CATEGORY_WEIGHTS.items():
        for kw, weight in weights.items():
            # 명사 빈도 기반 점수
            scores[cat] += fd[kw] * weight
            # 복합어가 명사 추출에서 쪼개지지 않는 경우를 보완하기 위해 원문 포함 여부도 약하게 반영
            if kw in text:
                scores[cat] += weight * 0.5

    return choose_best_category(scores)


## 방법 2: KoNLPy + TF-IDF (단어 가중치) 분류 함수

- 자주 나오지만 여러 문서에 흔한 단어는 낮게, 특정 문서에서만 자주 나오는 단어는 높게 가중치를 주는 방식

- TF  = 이 문서에서 단어가 나온 비율
- TF("회의", 메일A) = 메일A에서 "회의" 등장 횟수 / 메일A 전체 단어 수
- IDF = 전체 문서 중 이 단어가 없는 문서가 많을수록 높음
- 모든 문서에 나오는 단어는 IDF가 낮고, 특정 문서에만 나오는 단어는 IDF가 높아요.
- TF-IDF = TF × IDF


- 메일 텍스트에서 추출한 명사들의 TF-IDF 가중치
-    ↓
- 카테고리 키워드와 겹치는 단어의 가중치 합산
-  ↓
- 가중치 합이 가장 높은 카테고리로 분류

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def classify_tfidf(text, analyzer, subject="", filename=""):
    """
    명사 추출 → TF-IDF 수치화
    → 카테고리 키워드의 TF-IDF 가중치 합산
    → 제목/파일명 키워드 보정 후 가장 높은 카테고리로 분류
    """

    # 제목 태그 우선 확인
    for tag, cat in TAG_MAP.items():
        if tag in subject:
            return cat, 1.0

    nouns = extract_nouns(text, analyzer)
    if not nouns:
        return "기타", 0.0

    input_doc = " ".join(nouns)
    category_docs = {cat: " ".join(kws.keys()) for cat, kws in CATEGORY_WEIGHTS.items()}
    docs = [input_doc] + list(category_docs.values())

    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(docs)
    input_vec = tfidf_matrix[0].toarray()[0]
    feature_names = vectorizer.get_feature_names_out()

    scores = metadata_bonus(subject, filename)
    for cat, weights in CATEGORY_WEIGHTS.items():
        for kw, weight in weights.items():
            if kw in feature_names:
                idx = np.where(feature_names == kw)[0][0]
                scores[cat] += input_vec[idx] * weight
            # 복합어 보정
            if kw in text:
                scores[cat] += weight * 0.3

    return choose_best_category(scores)


## 분석기별 명사 추출 결과 비교

In [ ]:
# 실제 메일 텍스트로 분석기별 명사 추출 비교
print("=" * 60)
print("분석기별 명사 추출 결과 비교")
print("=" * 60)

for r in results[:15]:  # 앞 3개 메일만
    full_text = r["subject"] + " " + r["body"]
    print(f"\n📧 {r['subject'][:35]}")
    print(f"  Okt      : {extract_nouns(full_text, okt)}")
    print(f"  Komoran  : {extract_nouns(full_text, komoran)}")
    print(f"  Kkma     : {extract_nouns(full_text, kkma)}")
    print(f"  Hannanum : {extract_nouns(full_text, hannanum)}")

## 분석기 선택

In [ ]:
# 명사 추출 결과 비교를 통해 Okt 선택
# - Okt      : 정확한 명사 추출, 가장 빠름
# - Komoran  : 정확하나 Okt와 유사
# - Kkma     : 정확하나 처리 속도 느림
# - Hannanum : '요청드', '첨부파' 등 잘못된 형태소 추출

BEST_ANALYZER      = okt
BEST_ANALYZER_NAME = "Okt"
print(f"✅ 선택 분석기: {BEST_ANALYZER_NAME}")

## FreqDist / TF-IDF 분류 실행

In [ ]:
import time

ANALYZERS = {
    "Okt":      okt,
    "Komoran":  komoran,
    "Kkma":     kkma,
    "Hannanum": hannanum,
}

freqdist_results = {}
tfidf_results    = {}

for name, analyzer in ANALYZERS.items():
    for method, func, store in [
        ("FreqDist", classify_freqdist, freqdist_results),
        ("TF-IDF",   classify_tfidf,   tfidf_results),
    ]:
        print(f"⏳ [{method}] {name} 분석 중...")
        start = time.time()
        rows = []
        for r in results:
            subject   = r["subject"]
            sender    = r["sender"]
            body      = r["body"]
            pdf_text  = r["pdf_text"]
            pdf_name_text = get_pdf_filename_text(r.get("pdf_paths", []))
            full_text = subject + " " + body + " " + pdf_text + " " + pdf_name_text

            cat1, conf1 = func(subject, analyzer, subject=subject, filename=pdf_name_text)
            cat2, conf2 = func(subject + " " + body, analyzer, subject=subject, filename=pdf_name_text)
            cat3, conf3 = func(full_text, analyzer, subject=subject, filename=pdf_name_text)

            rows.append({
                "제목":           subject,
                "발신자":         sender,
                "분류_제목":      cat1,
                "분류_제목+본문": cat2,
                "분류_전체":      cat3,
                "신뢰도":         conf3,
                "회신":           detect_action(full_text),
                "기한":           find_deadline(full_text),
                "요약":           make_summary(body),
            })
        elapsed = round(time.time() - start, 2)
        store[name] = {"rows": rows, "time": elapsed}
        print(f"✅ {name} 완료 ({elapsed}초)")


## FreqDist vs TF-IDF 분류 결과 비교


In [ ]:
import pandas as pd

# Okt 기준으로 두 방법 결과 비교
comparison = []
for r, fd, tf in zip(
    results,
    freqdist_results["Okt"]["rows"],
    tfidf_results["Okt"]["rows"]
):
    comparison.append({
        "제목":          r["subject"][:25],
        "FreqDist 분류": fd["분류_전체"],
        "TF-IDF 분류":   tf["분류_전체"],
        "일치":          "✅" if fd["분류_전체"] == tf["분류_전체"] else "❌",
    })

pd.DataFrame(comparison)

## 처리시간 비교

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import numpy as np

matplotlib.rcParams["font.family"] = "Malgun Gothic"
matplotlib.rcParams["axes.unicode_minus"] = False

names   = list(ANALYZERS.keys())
fd_times = [freqdist_results[n]["time"] for n in names]
tf_times = [tfidf_results[n]["time"]    for n in names]

x     = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 4))
bars1 = ax.bar(x - width/2, fd_times, width, label="FreqDist", color="#3b82f6")
bars2 = ax.bar(x + width/2, tf_times, width, label="TF-IDF",   color="#10b981")

ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_title("분석기별 처리시간 비교 (FreqDist vs TF-IDF)")
ax.set_ylabel("초")
ax.legend()

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height()}초", ha="center", fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height()}초", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("results/method_compare.png", dpi=150)
plt.show()

## 최종 분류 결과 선택

In [ ]:
# 최종 분류 결과 선택
# PPT에서 사용한 최종 조합에 맞춰 Okt + TF-IDF를 사용
final_results = tfidf_results["Okt"]["rows"]
print(f"✅ 최종: Okt + TF-IDF, {len(final_results)}건")


---
# 5편. 워드클라우드 + PDF 이동 + CSV 저장

## 메일별 워드클라우드

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import os
import re
from wordcloud import WordCloud

matplotlib.rcParams["font.family"] = "Malgun Gothic"
matplotlib.rcParams["axes.unicode_minus"] = False

font_path = "C:/Windows/Fonts/malgun.ttf"

os.makedirs("results/wordcloud", exist_ok=True)

# 파일명에 사용할 수 없는 문자 제거
def clean_filename(text):
    text = re.sub(r'[\\/:*?"<>|]', '', str(text))
    text = re.sub(r'\s+', '_', text.strip())
    return text[:80]  # 파일명이 너무 길어지는 것 방지

for i, (r, row) in enumerate(zip(results, final_results)):
    full_text = r["subject"] + " " + r["body"] + " " + r["pdf_text"]

    nouns = extract_nouns(full_text, okt)

    if not nouns:
        print(f"⚠️ {i+1}. {r['subject'][:30]} - 명사 없음 스킵")
        continue

    wc = WordCloud(
        font_path=font_path,
        background_color="white",
        max_words=50,
        width=600,
        height=300,
        colormap="Blues",
    ).generate(" ".join(nouns))

    plt.figure(figsize=(10, 4))
    plt.imshow(wc)
    plt.axis("off")

    title = f"[{row['분류_전체']}] {r['subject'][:30]}"
    plt.title(title, fontsize=12, fontweight="bold")

    # PDF 제목 기준으로 저장
    pdf_name = r.get("pdf_filename", f"mail_{i+1:02d}")
    pdf_name = os.path.splitext(pdf_name)[0]
    pdf_name = clean_filename(pdf_name)

    filename = f"results/wordcloud/{pdf_name}.png"

    plt.savefig(filename, dpi=120, bbox_inches="tight")
    plt.show()

    print(f"✅ 저장 완료: {filename}")

## PDF 카테고리 폴더 이동

In [ ]:
import shutil

DOWNLOAD_DIR = "downloads"
categories = ["업무협조", "보고서", "회의록", "공지", "기타"]
for cat in categories:
    os.makedirs(os.path.join(DOWNLOAD_DIR, cat), exist_ok=True)

for r, row in zip(results, final_results):
    category  = row["분류_전체"]
    pdf_paths = r.get("pdf_paths", [])
    for src_path in pdf_paths:
        if not os.path.exists(src_path):
            print(f"⚠️ 파일 없음: {src_path}")
            continue
        filename = os.path.basename(src_path)
        dst_path = os.path.join(DOWNLOAD_DIR, category, filename)
        shutil.move(src_path, dst_path)
        print(f"📁 {filename} → {category}/")

print("✅ PDF 이동 완료")

## CSV 저장

In [ ]:
import pandas as pd

df = pd.DataFrame(final_results)
os.makedirs("results", exist_ok=True)

columns = ["제목", "발신자", "분류_전체", "회신", "기한", "요약"]
df[columns].to_csv("results/mail_classify_result.csv", index=False, encoding="utf-8-sig")
print("✅ CSV 저장 완료 → results/mail_classify_result.csv")

## 최종 요약 출력

In [ ]:
print("=" * 40)
print("최종 분류 결과 요약")
print("=" * 40)
print(df["분류_전체"].value_counts().to_string())
print()
print(f"전체 메일 수             : {len(df)}건")
print(f"회신 필요 메일           : {(df['회신'] == '필요').sum()}건")
print(f"기한 있는 메일           : {(df['기한'] != '-').sum()}건")


## 카테고리별 PDF 현황

In [ ]:
print("=" * 40)
print("카테고리별 PDF 현황")
print("=" * 40)

for cat in categories:
    folder = os.path.join(DOWNLOAD_DIR, cat)
    if os.path.exists(folder):
        files = [f for f in os.listdir(folder) if f.endswith(".pdf")]
        print(f"{cat:10} : {len(files)}개")
        for f in files:
            print(f"   └── {f}")
    else:
        print(f"{cat:10} : 폴더 없음")

In [ ]:
# 어떤 메일의 pdf_paths가 있는지 확인
for r in results:
    print(f"{r['subject'][:30]} → {r['pdf_paths']}")